In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Receiving objects: 100% (77/77), 1.05 MiB | 7.10 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [2]:
pip install ucimlrepo

In [3]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.9/206.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 9.8 MB/s eta 0:00:00


In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# Load dataset
breast_cancer = fetch_ucirepo(id=17)

X = breast_cancer.data.features
y = breast_cancer.data.targets

cancer_data = pd.concat([X, y], axis=1)

target_col = cancer_data.columns[-1]

# IMPORTANT: Do NOT fit generators on the full dataset
# Split first to avoid data leakage

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(cancer_data)

scores = {
    "CTABGAN": [],
    "WGAN_GP": [],
    "CTGAN": [],
    "CopulaGAN": [],
    "TVAE": [],
    "GaussianCopula": []
}

# Initialize dictionary to store all generated synthetic dataframes across all runs
synthetic_datasets = {
    "CTABGAN": [],
    "WGAN_GP": [],
    "CTGAN": [],
    "CopulaGAN": [],
    "TVAE": [],
    "GaussianCopula": []
}


N_RUNS = 10
N_SAMPLES = 1000
TEST_SIZE = 0.2

In [5]:
for seed in range(N_RUNS):

    print(f"\n================ RUN {seed+1}/{N_RUNS} ================")

    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # ---------------------------------------------------
    # TRAIN / TEST SPLIT (NO LEAKAGE)
    # ---------------------------------------------------

    train_real, test_real = train_test_split(
        cancer_data,
        test_size=0.2,
        stratify=cancer_data[target_col],
        random_state=seed
    )

    train_metadata = SingleTableMetadata()
    train_metadata.detect_from_dataframe(train_real)

    # ---------------------------------------------------
    # CTABGAN
    # ---------------------------------------------------

    try:

        data_path = f"breast_cancer_seed_{seed}.csv"
        train_real.to_csv(data_path, index=False)

        ctabgan = CTABGAN(
            raw_csv_path=data_path,
            categorical_columns=[target_col],
            log_columns=[],
            mixed_columns={},
            integer_columns=[],
            problem_type={"Classification": target_col}
        )

        ctabgan.fit()

        synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
            ctabgan.synthesizer.sample(N_SAMPLES)
        )

        # Store synthetic data for this run
        synthetic_datasets["CTABGAN"].append(synthetic_ctabgan.copy())

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_ctabgan,
            metadata=train_metadata
        )

        score = quality.get_score()

        scores["CTABGAN"].append(score)

        print("CTABGAN:", round(score, 4))

    except Exception as e:
        print("CTABGAN Failed:", e)


================ RUN 1/10 ================


100%|██████████| 150/150 [01:04<00:00,  2.34it/s]


Finished training in 76.76388192176819  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 403.37it/s]|
Column Shapes Score: 83.39%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 173.89it/s]|
Column Pair Trends Score: 80.33%

Overall Score (Average): 81.86%

CTABGAN: 0.8186

================ RUN 2/10 ================


100%|██████████| 150/150 [01:06<00:00,  2.26it/s]


Finished training in 72.59990525245667  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 416.00it/s]|
Column Shapes Score: 83.89%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 202.65it/s]|
Column Pair Trends Score: 78.65%

Overall Score (Average): 81.27%

CTABGAN: 0.8127

================ RUN 3/10 ================


...

Finished training in 68.53026986122131  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 479.98it/s]|
Column Shapes Score: 87.21%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 216.43it/s]|
Column Pair Trends Score: 81.01%

Overall Score (Average): 84.11%

CTABGAN: 0.8411

================ RUN 9/10 ================


100%|██████████| 150/150 [01:02<00:00,  2.41it/s]


Finished training in 67.73623752593994  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 368.03it/s]|
Column Shapes Score: 86.57%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 163.66it/s]|
Column Pair Trends Score: 76.15%

Overall Score (Average): 81.36%

CTABGAN: 0.8136

================ RUN 10/10 ================


100%|██████████| 150/150 [01:02<00:00,  2.41it/s]


Finished training in 67.53595614433289  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 443.23it/s]|
Column Shapes Score: 83.69%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 218.37it/s]|
Column Pair Trends Score: 80.22%

Overall Score (Average): 81.95%

CTABGAN: 0.8195


In [6]:
# ---------------------------------------------------
# WGAN-GP
# ---------------------------------------------------

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()

    data_wgan[target_col] = encoder.fit_transform(
        data_wgan[target_col]
    )

    scaler = StandardScaler()

    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    generator.train()
    critic.train()

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(
        critic,
        real_samples,
        fake_samples
    ):

        alpha = torch.rand(
            real_samples.size(0),
            1,
            device=device
        )

        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(
                critic_interpolates
            ),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(
            gradients.size(0),
            -1
        )

        gp = (
            (gradients.norm(2, dim=1) - 1) ** 2
        ).mean()

        return gp

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(
                    real_batch
                ).mean()

                critic_fake = critic(
                    fake_batch
                ).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = (
            generator(z)
            .cpu()
            .numpy()
        )

    synthetic = scaler.inverse_transform(
        synthetic_scaled
    )

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1) # Ensure values are 0 or 1 for inverse_transform
        .astype(int)
    )

    synthetic_wgan[target_col] = (
        encoder.inverse_transform(
            synthetic_wgan[target_col]
        )
    )

    # Store synthetic data for this run
    synthetic_datasets["WGAN_GP"].append(synthetic_wgan.copy())

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["WGAN_GP"].append(score)

    print("WGAN_GP:", round(score, 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 494.98it/s]|
Column Shapes Score: 89.44%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 214.28it/s]|
Column Pair Trends Score: 90.7%

Overall Score (Average): 90.07%

WGAN_GP: 0.9007


In [7]:
# ---------------------------------------------------
# SDV MODELS
# ---------------------------------------------------

models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(
            N_SAMPLES
        )

        # Store synthetic data for this run
        synthetic_datasets[model_name].append(synthetic_data.copy())

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        score = quality.get_score()

        scores[model_name].append(score)

        print(
            f"Run {seed+1}/{N_RUNS} | "
            f"{model_name}: {round(score,4)}"
        )

    except Exception as e:

        print(
            f"Run {seed+1}/{N_RUNS} | "
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 343.58it/s]|
Column Shapes Score: 69.44%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 215.37it/s]|
Column Pair Trends Score: 63.99%

Overall Score (Average): 66.71%

Run 10/10 | CTGAN: 0.6671
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 329.37it/s]|
Column Shapes Score: 68.45%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 217.03it/s]|
Column Pair Trends Score: 64.48%

Overall Score (Average): 66.46%

Run 10/10 | CopulaGAN: 0.6646
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 430.24it/s]|
Column Shapes Score: 86.45%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:02<00:00, 216.66it/s]|
Column Pair Trends Score: 86.04%

Overall Score (Average): 86.24%

Run 10/10 | TVAE: 0.8624
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31

In [8]:
# ==========================================================
# STORE ALL SYNTHETIC DATASETS GENERATED ACROSS 10 RUNS
# (Confirmation of collected data)
# ==========================================================

# The synthetic_datasets dictionary should now be populated from the generation loops.

print("--- Summary of Stored Synthetic Data Across Runs ---")
for model_name, datasets in synthetic_datasets.items():
    print(
        f"{model_name}: Stored {len(datasets)} datasets (expected {N_RUNS} datasets)."
    )
    if datasets:
        print(f"  Example head from first DataFrame for {model_name}:\n{datasets[0].head()}\n")
    else:
        print(f"  No datasets stored for {model_name}.\n")

# Now, synthetic_datasets is ready for TSTR evaluation in a subsequent cell if needed.
# X_real and y_real should be defined before any TSTR evaluation that uses them.

--- Summary of Stored Synthetic Data Across Runs ---
CTABGAN: Stored 10 datasets (expected 10 datasets).
  Example head from first DataFrame for CTABGAN:
     radius1   texture1  perimeter1        area1  smoothness1  compactness1  \
0  20.075084  21.538650   82.453684  1210.694806     0.113355      0.170255   
1  13.886153  17.749159  117.893200   435.422928     0.081899      0.063255   
2  13.631408  15.542875   71.365739   354.202796     0.069061      0.128348   
3  13.146513  13.455087   71.987063   723.921875     0.074583      0.037356   
4  19.533720  29.053488   89.529811   710.303105     0.121877      0.190127   

   concavity1  concave_points1  symmetry1  fractal_dimension1  ...   texture3  \
0    0.038671         0.023561   0.167793            0.065155  ...  33.573634   
1    0.005451         0.010697   0.148606            0.056781  ...  21.904632   
2    0.006557         0.011712   0.142399            0.062987  ...  24.023143   
3    0.036888         0.019516   0.149829      

In [9]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Re-define models to ensure it contains scikit-learn classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [10]:
# ----------------------------------------------------
# TRTR (Train Real, Test Real) Evaluation
# ----------------------------------------------------
print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")
trtr_results = []

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"  Running {model_name} for TRTR...")

    for seed in range(N_RUNS):

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(accuracy_score(y_test_real, y_pred))
        f1_scores.append(f1_score(y_test_real, y_pred, average="weighted", zero_division=0))
        precision_scores.append(precision_score(y_test_real, y_pred, average="weighted", zero_division=0))
        recall_scores.append(recall_score(y_test_real, y_pred, average="weighted", zero_division=0))

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": np.mean(accuracy_scores),
        "Accuracy Std_TRTR": np.std(accuracy_scores),
        "F1 Mean_TRTR": np.mean(f1_scores),
        "F1 Std_TRTR": np.std(f1_scores),
        "Precision Mean_TRTR": np.mean(precision_scores),
        "Precision Std_TRTR": np.std(precision_scores),
        "Recall Mean_TRTR": np.mean(recall_scores),
        "Recall Std_TRTR": np.std(recall_scores),
        "Accuracy (Mean±Std)_TRTR": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
        "F1 (Mean±Std)_TRTR": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
        "Precision (Mean±Std)_TRTR": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
        "Recall (Mean±Std)_TRTR": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)
display(trtr_results_df[
    [
        "Model",
        "Accuracy (Mean±Std)_TRTR",
        "F1 (Mean±Std)_TRTR",
        "Precision (Mean±Std)_TRTR",
        "Recall (Mean±Std)_TRTR"
    ]
])

--- Starting TRTR Evaluation (Train Real, Test Real) ---
  Running LogReg for TRTR...
  Running SVM-RBF for TRTR...
  Running KNN for TRTR...
  Running NaiveBayes for TRTR...
  Running DecisionTree for TRTR...
  Running RandomForest for TRTR...
  Running ExtraTrees for TRTR...
  Running GradientBoost for TRTR...
  Running AdaBoost for TRTR...
  Running MLP for TRTR...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
0,LogReg,0.9439 ± 0.0281,0.9435 ± 0.0285,0.9444 ± 0.0281,0.9439 ± 0.0281
1,SVM-RBF,0.9193 ± 0.0238,0.9172 ± 0.0251,0.9246 ± 0.0213,0.9193 ± 0.0238
2,KNN,0.9263 ± 0.0197,0.9256 ± 0.0204,0.9275 ± 0.0193,0.9263 ± 0.0197
3,NaiveBayes,0.9298 ± 0.0171,0.9291 ± 0.0174,0.9313 ± 0.0172,0.9298 ± 0.0171
4,DecisionTree,0.9175 ± 0.0226,0.9175 ± 0.0228,0.9184 ± 0.0231,0.9175 ± 0.0226
5,RandomForest,0.9500 ± 0.0176,0.9497 ± 0.0179,0.9507 ± 0.0171,0.9500 ± 0.0176
6,ExtraTrees,0.9500 ± 0.0180,0.9497 ± 0.0182,0.9505 ± 0.0179,0.9500 ± 0.0180
7,GradientBoost,0.9482 ± 0.0169,0.9479 ± 0.0171,0.9489 ± 0.0167,0.9482 ± 0.0169
8,AdaBoost,0.9553 ± 0.0114,0.9549 ± 0.0116,0.9563 ± 0.0108,0.9553 ± 0.0114
9,MLP,0.9342 ± 0.0278,0.9330 ± 0.0296,0.9368 ± 0.0240,0.9342 ± 0.0278


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

def evaluate_models(train_df, test_df, label_col, models, test_size=0.2, seed=42):
    # Split TRAIN df -> (train part only)  (we don’t need train_df's test split)
    X_train = train_df.drop(columns=[label_col])
    y_train = train_df[label_col]

    X_train, _, y_train, _ = train_test_split(
        X_train, y_train, test_size=test_size, random_state=seed, stratify=y_train
    )

    # Split TEST df -> (test part only)
    X_test = test_df.drop(columns=[label_col])
    y_test = test_df[label_col]

    _, X_test, _, y_test = train_test_split(
        X_test, y_test, test_size=test_size, random_state=seed, stratify=y_test
    )

    # Scale using TRAIN statistics only
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s  = scaler.transform(X_test)

    rows = []
    for name, clf in models.items():
        clf.fit(X_train_s, y_train)

        y_pred = clf.predict(X_test_s)

        # AUC needs probabilities (or decision_function); handle both safely
        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(X_test_s)[:, 1]
        elif hasattr(clf, "decision_function"):
            scores = clf.decision_function(X_test_s)
            # convert to 0-1-ish range (not perfect prob, but works for AUC)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        else:
            y_prob = None

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)
        precision = precision_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)

        auc = roc_auc_score(y_test, y_prob) if y_prob is not None else float("nan")

        rows.append({"Model": name, "Accuracy": acc, "F1": f1, "Precision": precision, "Recall": recall, "AUC": auc})

    return pd.DataFrame(rows).sort_values(by="AUC", ascending=False)

In [19]:
import pandas as pd
import numpy as np # Need numpy for mean/std

label_col = "Diagnosis"
model_order = ["CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "WGAN_GP", "CTABGAN"]

# The existing trtr_results calculation (single evaluation)
trtr_results = evaluate_models(
    train_df=cancer_data,
    test_df= cancer_data,
    label="diagnosis",
    models=models
)

print("TRTR (Train Real, Test Real)")
display(trtr_results)
print("=" * 70)

all_comparisons = []

for synth_name in model_order:
    print(f"{synth_name} - TSTR (Train Synthetic, Test Real) Evaluation Across {N_RUNS} Runs")

    # Dictionary to collect results for each classifier across all N_RUNS for the current synth_name
    # Each key (classifier name) will hold a list of scores for each metric (from each run)
    classifier_run_metrics = {
        clf_name: {"Accuracy": [], "F1": [], "Precision": [], "Recall": [], "AUC": []}
        for clf_name in models.keys()
    }

    # Iterate through each synthetic dataset generated for this synthetic model across N_RUNS
    # `synthetic_datasets[synth_name]` is a list of N_RUNS dataframes
    for run_idx, synthetic_train_df_for_run in enumerate(synthetic_datasets[synth_name]):
        # Evaluate models using the synthetic data from one specific run as training data
        # and the real data (split internally by evaluate_models) as test data.
        run_results = evaluate_models(
            train_df=synthetic_train_df_for_run, # This is now a single DataFrame
            test_df=cancer_data,                 # Full real data for internal splitting
            label="diagnosis",
            models=models,
            test_size=TEST_SIZE,
            seed=run_idx                         # Use run_idx for seed for reproducibility of internal splits
        )

        # Collect metrics for each classifier from this run
        for _, row in run_results.iterrows():
            clf_name = row["Model"]
            classifier_run_metrics[clf_name]["Accuracy"].append(row["Accuracy"])
            classifier_run_metrics[clf_name]["F1"].append(row["F1"])
            classifier_run_metrics[clf_name]["Precision"].append(row["Precision"])
            classifier_run_metrics[clf_name]["Recall"].append(row["Recall"])
            classifier_run_metrics[clf_name]["AUC"].append(row["AUC"])

    # Calculate the mean of the metrics across all N_RUNS for each classifier for the current synthetic model
    tstr_mean_metrics = []
    for clf_name, metrics_list in classifier_run_metrics.items():
        tstr_mean_metrics.append({
            "Model": clf_name,
            "Accuracy": np.mean(metrics_list["Accuracy"]),
            "F1": np.mean(metrics_list["F1"]),
            "Precision": np.mean(metrics_list["Precision"]),
            "Recall": np.mean(metrics_list["Recall"]),
            "AUC": np.mean(metrics_list["AUC"])
        })

    # Convert to DataFrame for display and comparison
    tstr_results = pd.DataFrame(tstr_mean_metrics).sort_values(by="AUC", ascending=False)


    print(f"{synth_name} - TSTR (Train Synthetic, Test Real)")
    display(tstr_results)

    comparison = trtr_results.merge(
        tstr_results, on="Model", suffixes=('_TRTR', '_TSTR')
    )

    comparison["AUC_Drop"] = comparison["AUC_TRTR"] - comparison["AUC_TSTR"]

    if "F1_TRTR" in comparison.columns and "F1_TSTR" in comparison.columns:
        comparison["F1_Drop"] = comparison["F1_TRTR"] - comparison["F1_TSTR"]
    if "Accuracy_TRTR" in comparison.columns and "Accuracy_TSTR" in comparison.columns:
        comparison["Accuracy_Drop"] = comparison["Accuracy_TRTR"] - comparison["Accuracy_TSTR"]

    comparison["Synthetic_Model"] = synth_name
    comparison = comparison.sort_values("AUC_Drop", ascending=False)

    print(f"{synth_name} - TRTR vs TSTR Comparison")
    display(comparison)
    print("=" * 70)

    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)

summary = (combined_comparison
           .groupby("Synthetic_Model", as_index=False)["AUC_Drop"]
           .mean()
           .sort_values("AUC_Drop"))

print("Average AUC drop by synthetic generator (lower is better):")
display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy,F1,Precision,Recall,AUC
6,ExtraTrees,0.973684,0.962963,1.000000,0.928571,0.998843
0,LogReg,0.973684,0.963855,0.975610,0.952381,0.996032
7,GradientBoost,0.964912,0.950000,1.000000,0.904762,0.994709
1,SVM-RBF,0.973684,0.962963,1.000000,0.928571,0.994709
9,MLP,0.964912,0.950000,1.000000,0.904762,0.993717
5,RandomForest,0.973684,0.962963,1.000000,0.928571,0.992890
3,NaiveBayes,0.921053,0.888889,0.923077,0.857143,0.989087
8,AdaBoost,0.982456,0.975610,1.000000,0.952381,0.984127
2,KNN,0.956140,0.938272,0.974359,0.904762,0.982308
4,DecisionTree,0.929825,0.904762,0.904762,0.904762,0.924603


CTGAN - TSTR (Train Synthetic, Test Real) Evaluation Across 10 Runs
CTGAN - TSTR (Train Synthetic, Test Real)


,Model,Accuracy,F1,Precision,Recall,AUC
2,KNN,0.491228,0.472727,0.382353,0.619048,0.578208
9,MLP,0.429825,0.511278,0.373626,0.809524,0.532738
7,GradientBoost,0.456140,0.483333,0.371795,0.690476,0.500331
5,RandomForest,0.421053,0.400000,0.323529,0.523810,0.446263
6,ExtraTrees,0.403509,0.403509,0.319444,0.547619,0.425595
8,AdaBoost,0.385965,0.339623,0.281250,0.428571,0.416005
4,DecisionTree,0.368421,0.357143,0.285714,0.476190,0.390873
3,NaiveBayes,0.324561,0.421053,0.307692,0.666667,0.343585
1,SVM-RBF,0.385965,0.500000,0.357143,0.833333,0.230159
0,LogReg,0.307018,0.423358,0.305263,0.690476,0.226521


CTGAN - TRTR vs TSTR Comparison


...

,Model,Accuracy,F1,Precision,Recall,AUC
6,ExtraTrees,0.906140,0.862994,0.942888,0.802381,0.966386
0,LogReg,0.881579,0.822414,0.915050,0.752381,0.962169
3,NaiveBayes,0.879825,0.825947,0.884314,0.780952,0.953208
5,RandomForest,0.893860,0.845489,0.923706,0.785714,0.952364
8,AdaBoost,0.873684,0.808664,0.893647,0.750000,0.950231
1,SVM-RBF,0.879825,0.822144,0.905606,0.759524,0.948991
7,GradientBoost,0.873684,0.814903,0.881775,0.766667,0.942593
9,MLP,0.855263,0.785153,0.871680,0.723810,0.919841
2,KNN,0.873684,0.806921,0.922216,0.723810,0.900876
4,DecisionTree,0.755263,0.654769,0.697902,0.628571,0.728869


CTABGAN - TRTR vs TSTR Comparison


,Model,Accuracy_TRTR,F1_TRTR,Precision_TRTR,Recall_TRTR,AUC_TRTR,Accuracy_TSTR,F1_TSTR,Precision_TSTR,Recall_TSTR,AUC_TSTR,AUC_Drop,F1_Drop,Accuracy_Drop,Synthetic_Model
9,DecisionTree,0.929825,0.904762,0.904762,0.904762,0.924603,0.755263,0.654769,0.697902,0.628571,0.728869,0.195734,0.249993,0.174561,CTABGAN
8,KNN,0.956140,0.938272,0.974359,0.904762,0.982308,0.873684,0.806921,0.922216,0.723810,0.900876,0.081432,0.131350,0.082456,CTABGAN
4,MLP,0.964912,0.950000,1.000000,0.904762,0.993717,0.855263,0.785153,0.871680,0.723810,0.919841,0.073876,0.164847,0.109649,CTABGAN
2,GradientBoost,0.964912,0.950000,1.000000,0.904762,0.994709,0.873684,0.814903,0.881775,0.766667,0.942593,0.052116,0.135097,0.091228,CTABGAN
3,SVM-RBF,0.973684,0.962963,1.000000,0.928571,0.994709,0.879825,0.822144,0.905606,0.759524,0.948991,0.045718,0.140819,0.093860,CTABGAN
5,RandomForest,0.973684,0.962963,1.000000,0.928571,0.992890,0.893860,0.845489,0.923706,0.785714,0.952364,0.040526,0.117474,0.079825,CTABGAN
6,NaiveBayes,0.921053,0.888889,0.923077,0.857143,0.989087,0.879825,0.825947,0.884314,0.780952,0.953208,0.035880,0.062942,0.041228,CTABGAN
7,AdaBoost,0.982456,0.975610,1.000000,0.952381,0.984127,0.873684,0.808664,0.893647,0.750000,0.950231,0.033896,0.166946,0.108772,CTABGAN
1,LogReg,0.973684,0.963855,0.975610,0.952381,0.996032,0.881579,0.822414,0.915050,0.752381,0.962169,0.033862,0.141441,0.092105,CTABGAN
0,ExtraTrees,0.973684,0.962963,1.000000,0.928571,0.998843,0.906140,0.862994,0.942888,0.802381,0.966386,0.032457,0.099969,0.067544,CTABGAN


Average AUC drop by synthetic generator (lower is better):


,Synthetic_Model,AUC_Drop
5,WGAN_GP,0.015526
4,TVAE,0.019808
0,CTABGAN,0.062550
3,GaussianCopula,0.073595
2,CopulaGAN,0.169180
1,CTGAN,0.576075


In [22]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Re-define models to ensure it contains scikit-learn classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

# Assume cancer_data and target_col are already defined globally
X_real = cancer_data.drop(columns=[target_col])
y_real = cancer_data[target_col]

N_RUNS = 10 # Number of runs for evaluation, consistent with prior cells
TEST_SIZE = 0.3 # Test set size for train_test_split

# ----------------------------------------------------
# TRTR (Train Real, Test Real) Evaluation
# ----------------------------------------------------
print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")
trtr_results = []

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"  Running {model_name} for TRTR...")

    for seed in range(N_RUNS):

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X_real,
            y_real,
            test_size=TEST_SIZE,
            stratify=y_real,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(accuracy_score(y_test_real, y_pred))
        f1_scores.append(f1_score(y_test_real, y_pred, average="weighted", zero_division=0))
        precision_scores.append(precision_score(y_test_real, y_pred, average="weighted", zero_division=0))
        recall_scores.append(recall_score(y_test_real, y_pred, average="weighted", zero_division=0))

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": np.mean(accuracy_scores),
        "Accuracy Std_TRTR": np.std(accuracy_scores),
        "F1 Mean_TRTR": np.mean(f1_scores),
        "F1 Std_TRTR": np.std(f1_scores),
        "Precision Mean_TRTR": np.mean(precision_scores),
        "Precision Std_TRTR": np.std(precision_scores),
        "Recall Mean_TRTR": np.mean(recall_scores),
        "Recall Std_TRTR": np.std(recall_scores),
        "Accuracy (Mean±Std)_TRTR": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
        "F1 (Mean±Std)_TRTR": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
        "Precision (Mean±Std)_TRTR": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
        "Recall (Mean±Std)_TRTR": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)
display(trtr_results_df[
    [
        "Model",
        "Accuracy (Mean±Std)_TRTR",
        "F1 (Mean±Std)_TRTR",
        "Precision (Mean±Std)_TRTR",
        "Recall (Mean±Std)_TRTR"
    ]
])


--- Starting TRTR Evaluation (Train Real, Test Real) ---
  Running LogReg for TRTR...
  Running SVM-RBF for TRTR...
  Running KNN for TRTR...
  Running NaiveBayes for TRTR...
  Running DecisionTree for TRTR...
  Running RandomForest for TRTR...
  Running ExtraTrees for TRTR...
  Running GradientBoost for TRTR...
  Running AdaBoost for TRTR...
  Running MLP for TRTR...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
0,LogReg,0.9468 ± 0.0170,0.9466 ± 0.0171,0.9472 ± 0.0172,0.9468 ± 0.0170
1,SVM-RBF,0.9111 ± 0.0113,0.9090 ± 0.0118,0.9165 ± 0.0117,0.9111 ± 0.0113
2,KNN,0.9281 ± 0.0150,0.9276 ± 0.0151,0.9287 ± 0.0157,0.9281 ± 0.0150
3,NaiveBayes,0.9310 ± 0.0163,0.9303 ± 0.0164,0.9327 ± 0.0168,0.9310 ± 0.0163
4,DecisionTree,0.9158 ± 0.0266,0.9159 ± 0.0264,0.9168 ± 0.0260,0.9158 ± 0.0266
5,RandomForest,0.9491 ± 0.0187,0.9489 ± 0.0188,0.9496 ± 0.0185,0.9491 ± 0.0187
6,ExtraTrees,0.9538 ± 0.0149,0.9536 ± 0.0151,0.9541 ± 0.0147,0.9538 ± 0.0149
7,GradientBoost,0.9485 ± 0.0122,0.9483 ± 0.0123,0.9493 ± 0.0122,0.9485 ± 0.0122
8,AdaBoost,0.9515 ± 0.0148,0.9512 ± 0.0149,0.9520 ± 0.0146,0.9515 ± 0.0148
9,MLP,0.9392 ± 0.0162,0.9387 ± 0.0164,0.9403 ± 0.0160,0.9392 ± 0.0162


In [25]:
# ----------------------------------------------------
# TSTR (Train Synthetic, Test Real) Evaluation
# (Without AUC Metric)
# ----------------------------------------------------
print("\n--- Starting TSTR Evaluation (Train Synthetic, Test Real) ---")

all_tstr_results = []

if 'synthetic_datasets' not in locals() and 'synthetic_datasets' not in globals():
    print("Warning: 'synthetic_datasets' variable not found. TSTR evaluation will be skipped.")

elif not synthetic_datasets:
    print("Warning: 'synthetic_datasets' is empty. TSTR evaluation will be skipped.")

else:
    for synth_data_name, list_of_synthetic_dfs in synthetic_datasets.items():

        print(f"\nEvaluating TSTR for: {synth_data_name}")

        classifier_run_metrics = {
            clf_name: {
                "Accuracy": [],
                "F1": [],
                "Precision": [],
                "Recall": []
            }
            for clf_name in models.keys()
        }

        for run_idx, synthetic_train_df in enumerate(list_of_synthetic_dfs):

            run_results_df = evaluate_models(
                train_df=synthetic_train_df,
                test_df=cancer_data,
                label_col=target_col,
                models=models,
                test_size=TEST_SIZE,
                seed=run_idx
            )

            for _, row in run_results_df.iterrows():
                clf_name = row["Model"]

                classifier_run_metrics[clf_name]["Accuracy"].append(row["Accuracy"])
                classifier_run_metrics[clf_name]["F1"].append(row["F1"])
                classifier_run_metrics[clf_name]["Precision"].append(row["Precision"])
                classifier_run_metrics[clf_name]["Recall"].append(row["Recall"])

        # Aggregate Mean ± Std across runs
        for clf_name, metrics in classifier_run_metrics.items():

            acc_mean = np.mean(metrics["Accuracy"])
            acc_std = np.std(metrics["Accuracy"])

            f1_mean = np.mean(metrics["F1"])
            f1_std = np.std(metrics["F1"])

            prec_mean = np.mean(metrics["Precision"])
            prec_std = np.std(metrics["Precision"])

            rec_mean = np.mean(metrics["Recall"])
            rec_std = np.std(metrics["Recall"])

            all_tstr_results.append({
                "Synthetic_Model": synth_data_name,
                "Model": clf_name,

                "Accuracy Mean_TSTR": acc_mean,
                "Accuracy Std_TSTR": acc_std,

                "F1 Mean_TSTR": f1_mean,
                "F1 Std_TSTR": f1_std,

                "Precision Mean_TSTR": prec_mean,
                "Precision Std_TSTR": prec_std,

                "Recall Mean_TSTR": rec_mean,
                "Recall Std_TSTR": rec_std,

                "Accuracy (Mean±Std)_TSTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
                "F1 (Mean±Std)_TSTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
                "Precision (Mean±Std)_TSTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
                "Recall (Mean±Std)_TSTR": f"{rec_mean:.4f} ± {rec_std:.4f}",
            })

    tstr_results_df = pd.DataFrame(all_tstr_results)

    display(
        tstr_results_df[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy (Mean±Std)_TSTR",
                "F1 (Mean±Std)_TSTR",
                "Precision (Mean±Std)_TSTR",
                "Recall (Mean±Std)_TSTR",
            ]
        ].sort_values(["Synthetic_Model", "Model"])
    )


--- Starting TSTR Evaluation (Train Synthetic, Test Real) ---

Evaluating TSTR for: CTABGAN

Evaluating TSTR for: WGAN_GP

Evaluating TSTR for: CTGAN

Evaluating TSTR for: CopulaGAN

Evaluating TSTR for: TVAE

Evaluating TSTR for: GaussianCopula


,Synthetic_Model,Model,Accuracy (Mean±Std)_TSTR,F1 (Mean±Std)_TSTR,Precision (Mean±Std)_TSTR,Recall (Mean±Std)_TSTR
8,CTABGAN,AdaBoost,0.8661 ± 0.0394,0.8026 ± 0.0674,0.8832 ± 0.0554,0.7438 ± 0.1085
4,CTABGAN,DecisionTree,0.6971 ± 0.0934,0.5906 ± 0.1156,0.6146 ± 0.1205,0.5906 ± 0.1482
6,CTABGAN,ExtraTrees,0.8977 ± 0.0211,0.8529 ± 0.0306,0.9270 ± 0.0571,0.7953 ± 0.0603
7,CTABGAN,GradientBoost,0.8719 ± 0.0220,0.8196 ± 0.0339,0.8712 ± 0.0616,0.7812 ± 0.0688
2,CTABGAN,KNN,0.8702 ± 0.0206,0.8048 ± 0.0361,0.9198 ± 0.0488,0.7203 ± 0.0642
0,CTABGAN,LogReg,0.8784 ± 0.0194,0.8201 ± 0.0348,0.9176 ± 0.0523,0.7469 ± 0.0640
9,CTABGAN,MLP,0.8614 ± 0.0303,0.7957 ± 0.0527,0.8850 ± 0.0619,0.7328 ± 0.0972
3,CTABGAN,NaiveBayes,0.8766 ± 0.0226,0.8232 ± 0.0370,0.8862 ± 0.0436,0.7734 ± 0.0671
5,CTABGAN,RandomForest,0.8977 ± 0.0225,0.8545 ± 0.0290,0.9229 ± 0.0662,0.8016 ± 0.0546
1,CTABGAN,SVM-RBF,0.8836 ± 0.0226,0.8331 ± 0.0344,0.9012 ± 0.0551,0.7797 ± 0.0611
